# SMA/ALS scRNA-seq Datasets: Unified Preprocessing

Download and unify six GEO datasets into a common `AnnData` schema, then save one `.h5ad` per GSE.

**Datasets**

| GSE | Format | Samples | Has cell-type? |
|-----|--------|---------|----------------|
| GSE287569 | 10x H5 (per-sample) | 12 | no |
| GSE173524 | combined UMI tsv + nuclei metadata | combined | **yes** |
| GSE167332 | 6 dense sample matrices + 1 MTX | 7 | no |
| GSE219201 | 10x MTX (per-sample) | 6 | no |
| GSE242942 | 10x MTX (per-sample) | 2 | no |
| GSE208629 | 10x MTX (per-sample) | 2 | no |

**Unified schema**
- `adata.var_names`: gene symbol (fallback: ensembl id)
- `adata.var['gene_symbol']`, `adata.var['gene_symbol_upper']`, `adata.var['ensembl_id']` (if available)
- `adata.obs_names`: `{sample_id}_{barcode}` (globally unique)
- `adata.obs['cell_id']`: same as `obs_names` (explicit column)
- `adata.obs['cell_type']`: cell type if available, otherwise `'unknown'`
- `adata.obs['sample_id']`, `adata.obs['gse_id']`, `adata.obs['condition']`
- `adata.X`: raw counts (CSR sparse)

## 1. Install & import

In [ ]:
# Colab: install scanpy + anndata stack
%pip install -q scanpy anndata h5py scipy pandas numpy tqdm

In [ ]:
import os
import re
import gzip
import tarfile
import shutil
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.io as sio
import scanpy as sc
import anndata as ad
from tqdm.auto import tqdm

sc.settings.verbosity = 1

DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/output')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, '\noutput:', OUTPUT_DIR)

## 2. Generic helpers — download, extract, standardise

In [ ]:
def gse_ftp_url(gse_id: str) -> str:
    """GEO FTP base URL for a series, e.g. GSE287569 -> .../GSE287nnn/GSE287569/suppl/."""
    stub = gse_id[:-3] + 'nnn'
    return f'https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse_id}/suppl/'


def download(url: str, dest: Path, overwrite: bool = False) -> Path:
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not overwrite:
        return dest
    tmp = dest.with_suffix(dest.suffix + '.part')
    print(f'  download {url} -> {dest.name}')
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    return dest


def download_suppl(gse_id: str, fname: str, gse_dir: Path) -> Path:
    return download(gse_ftp_url(gse_id) + fname, gse_dir / fname)


def extract_tar(tar_path: Path, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path) as tf:
        tf.extractall(out_dir)
    return out_dir


def prepare_gse(gse_id: str) -> Path:
    """Download GSE_RAW.tar and extract; return the directory with the extracted files."""
    gse_dir = DATA_DIR / gse_id
    gse_dir.mkdir(parents=True, exist_ok=True)
    tar_name = f'{gse_id}_RAW.tar'
    tar_path = download_suppl(gse_id, tar_name, gse_dir)
    extracted = gse_dir / 'extracted'
    if not extracted.exists():
        extract_tar(tar_path, extracted)
    return extracted

In [ ]:
# ---- AnnData unification helpers ----

_CELLTYPE_KEYS = [
    'cell_type', 'celltype', 'cell.type', 'CellType', 'Cell_Type', 'Cell.Type',
    'cell_class', 'class', 'subtype', 'cell_subtype',
    'annotation', 'Annotation', 'annot', 'cluster_label', 'cell_label',
    'seurat_clusters', 'cluster', 'clusters', 'leiden', 'louvain',
]

_GENE_SYM_KEYS = ['gene_symbol', 'gene_symbols', 'symbol', 'Symbol', 'gene_name', 'GeneSymbol']
_GENE_ID_KEYS  = ['gene_ids', 'gene_id', 'ensembl_id', 'EnsemblID', 'ensembl']


def _first_match(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def standardize_obs(adata, gse_id, sample_id=None, condition=None,
                    cell_type_col=None):
    """Add gse_id/sample_id/condition/cell_type; normalize cell_id."""
    adata.obs['gse_id'] = gse_id
    if sample_id is not None:
        adata.obs['sample_id'] = str(sample_id)
    elif 'sample_id' not in adata.obs.columns:
        adata.obs['sample_id'] = 'unknown'
    adata.obs['condition'] = str(condition) if condition is not None else 'unknown'

    found = cell_type_col or _first_match(adata.obs.columns, _CELLTYPE_KEYS)
    if found and found != 'cell_type':
        adata.obs['cell_type'] = adata.obs[found].astype(str)
    elif 'cell_type' not in adata.obs.columns:
        adata.obs['cell_type'] = 'unknown'
    else:
        adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)

    if sample_id is not None:
        new_names = [f'{sample_id}_{b}' for b in adata.obs_names]
        adata.obs_names = pd.Index(new_names)
    adata.obs['cell_id'] = adata.obs_names
    return adata


def standardize_var(adata):
    """Pick a gene_symbol column; ensure var_names = symbol when possible."""
    cols = list(adata.var.columns)
    sym_col = _first_match(cols, _GENE_SYM_KEYS)
    id_col  = _first_match(cols, _GENE_ID_KEYS)

    if sym_col:
        symbols = adata.var[sym_col].astype(str)
    else:
        # Heuristic: if current var_names look like ENSEMBL ids and there is no symbol, keep ids
        symbols = pd.Series(adata.var_names.astype(str), index=adata.var_names)

    if id_col:
        adata.var['ensembl_id'] = adata.var[id_col].astype(str)
    elif adata.var_names.astype(str).str.match(r'^ENS[A-Z]*G\d+').all():
        adata.var['ensembl_id'] = adata.var_names.astype(str)

    adata.var['gene_symbol'] = symbols.values
    adata.var['gene_symbol_upper'] = adata.var['gene_symbol'].astype(str).str.upper()

    use_as_names = adata.var['gene_symbol'].astype(str)
    use_as_names = use_as_names.where(~use_as_names.isin(['', 'nan', 'NaN', 'None']),
                                       pd.Series(adata.var_names.astype(str)).values)
    adata.var_names = pd.Index(use_as_names.values)
    adata.var_names_make_unique()
    return adata


def ensure_csr(adata):
    if not sp.issparse(adata.X):
        adata.X = sp.csr_matrix(adata.X)
    elif not sp.isspmatrix_csr(adata.X):
        adata.X = adata.X.tocsr()
    return adata


def finalize(adata, gse_id):
    standardize_var(adata)
    if 'cell_id' not in adata.obs.columns:
        adata.obs['cell_id'] = adata.obs_names
    adata.obs['gse_id'] = gse_id
    for col in ('sample_id', 'condition', 'cell_type'):
        if col not in adata.obs.columns:
            adata.obs[col] = 'unknown'
        adata.obs[col] = adata.obs[col].astype(str)
    ensure_csr(adata)
    keep = ['cell_id', 'gse_id', 'sample_id', 'condition', 'cell_type']
    rest = [c for c in adata.obs.columns if c not in keep]
    adata.obs = adata.obs[keep + rest]
    return adata


def report(adata, gse_id):
    print(f'\n[{gse_id}] cells={adata.n_obs:,}  genes={adata.n_vars:,}')
    print('  samples :', adata.obs["sample_id"].nunique(),
          '| conditions :', sorted(adata.obs["condition"].unique()))
    print('  cell_types :', adata.obs["cell_type"].nunique(),
          ' top5:', adata.obs["cell_type"].value_counts().head().to_dict())
    print('  X dtype/sparse :', adata.X.dtype, sp.issparse(adata.X))

## 3. GSE287569 — 12× 10x H5
Filenames look like `GSM8748142_SOD1_s1_filtered_feature_bc_matrix.h5`. We use `sc.read_10x_h5` per file and concatenate.

In [ ]:
def load_GSE287569():
    gse = 'GSE287569'
    src = prepare_gse(gse)
    h5s = sorted(src.rglob('*_filtered_feature_bc_matrix.h5'))
    assert h5s, f'no h5 found under {src}'
    parts = []
    for p in tqdm(h5s, desc=gse):
        m = re.match(r'(GSM\d+)_(.+)_filtered_feature_bc_matrix\.h5', p.name)
        gsm, label = (m.group(1), m.group(2)) if m else (p.stem, p.stem)
        a = sc.read_10x_h5(str(p))
        a.var_names_make_unique()
        standardize_obs(a, gse, sample_id=gsm, condition=label)
        parts.append(a)
    adata = ad.concat(parts, join='outer', merge='first', index_unique=None)
    return finalize(adata, gse)

## 4. GSE173524 — combined UMI tsv + nuclei metadata
Uses the pre-combined `GSE173524_umi.tsv.gz` (raw UMI counts; rows = genes, cols = cells per Seurat convention) and `GSE173524_metadata.nuclei.tsv.gz` (per-nucleus metadata including cell-type/HTO assignments).

In [ ]:
def load_GSE173524():
    gse = 'GSE173524'
    gse_dir = DATA_DIR / gse
    umi_path  = download_suppl(gse, f'{gse}_umi.tsv.gz', gse_dir)
    meta_path = download_suppl(gse, f'{gse}_metadata.nuclei.tsv.gz', gse_dir)
    sample_meta_path = download_suppl(gse, f'{gse}_metadata.samples.tsv.gz', gse_dir)

    print('  reading UMI tsv ...')
    umi = pd.read_csv(umi_path, sep='\t', index_col=0)  # genes x cells
    meta = pd.read_csv(meta_path, sep='\t', index_col=0)
    sample_meta = pd.read_csv(sample_meta_path, sep='\t')

    # genes x cells -> AnnData (cells x genes)
    X = sp.csr_matrix(umi.T.values)
    adata = ad.AnnData(X=X,
                       obs=pd.DataFrame(index=umi.columns.astype(str)),
                       var=pd.DataFrame(index=umi.index.astype(str)))
    adata.var['gene_symbol'] = adata.var_names.astype(str)

    # attach metadata
    common = adata.obs_names.intersection(meta.index.astype(str))
    print(f'  meta rows: {len(meta):,}, matched to UMI columns: {len(common):,} / {adata.n_obs:,}')
    meta = meta.loc[common]
    adata = adata[common].copy()
    for c in meta.columns:
        adata.obs[c] = meta[c].astype(str).values

    # sample column heuristic (HTO_maxID or similar)
    sample_col = _first_match(adata.obs.columns,
                              ['sample', 'Sample', 'orig.ident', 'HTO_maxID', 'hash.ID', 'sample_id'])
    adata.obs['sample_id'] = adata.obs[sample_col].astype(str) if sample_col else 'GSE173524'

    # condition heuristic (treatment / genotype / condition)
    cond_col = _first_match(adata.obs.columns,
                            ['condition', 'Condition', 'genotype', 'Genotype', 'group', 'Group',
                             'treatment', 'Treatment'])
    if cond_col:
        adata.obs['condition'] = adata.obs[cond_col].astype(str)

    standardize_obs(adata, gse, sample_id=None)  # keep sample_id already set
    return finalize(adata, gse)

## 5. GSE167332 — 6 dense plate matrices + 1 MTX bundle (heterogeneous)
Six `Sample{N}_matrix.txt.gz` are dense gene×cell tables; one extra GSM is in CellRanger-style MTX with separate row/col name files.

In [ ]:
def _read_dense_matrix(path: Path) -> ad.AnnData:
    """Read a gene x cell dense tsv/txt(.gz). Returns AnnData (cells x genes)."""
    df = pd.read_csv(path, sep='\t', index_col=0)
    a = ad.AnnData(X=sp.csr_matrix(df.T.values),
                   obs=pd.DataFrame(index=df.columns.astype(str)),
                   var=pd.DataFrame(index=df.index.astype(str)))
    a.var['gene_symbol'] = a.var_names.astype(str)
    return a


def load_GSE167332():
    gse = 'GSE167332'
    src = prepare_gse(gse)
    parts = []

    # Dense plate matrices
    for p in sorted(src.rglob('GSM*_Sample*_matrix.txt.gz')):
        m = re.match(r'(GSM\d+)_(Sample\d+)_matrix\.txt\.gz', p.name)
        gsm, label = (m.group(1), m.group(2)) if m else (p.stem, p.stem)
        print(f'  dense: {p.name}')
        a = _read_dense_matrix(p)
        standardize_obs(a, gse, sample_id=gsm, condition=label)
        parts.append(a)

    # MTX bundle (counts.mtx + rownames + colnames)
    mtx_files = sorted(src.rglob('GSM*_counts.mtx.gz'))
    for mtx_path in mtx_files:
        gsm = mtx_path.name.split('_')[0]
        rownames = mtx_path.with_name(mtx_path.name.replace('.mtx.gz', '.mtx.rownames.txt.gz'))
        colnames = mtx_path.with_name(mtx_path.name.replace('.mtx.gz', '.mtx.colnames.txt.gz'))
        print(f'  mtx  : {mtx_path.name}')
        with gzip.open(mtx_path, 'rb') as fh:
            X = sio.mmread(fh).tocsr()
        rows = pd.read_csv(rownames, sep='\t', header=None)[0].astype(str).tolist()
        cols = pd.read_csv(colnames, sep='\t', header=None)[0].astype(str).tolist()
        # Orient: assume rows=genes, cols=cells (CellRanger convention)
        if X.shape == (len(rows), len(cols)):
            a = ad.AnnData(X=X.T.tocsr(),
                           obs=pd.DataFrame(index=cols),
                           var=pd.DataFrame(index=rows))
        else:
            a = ad.AnnData(X=X.tocsr(),
                           obs=pd.DataFrame(index=rows),
                           var=pd.DataFrame(index=cols))
        a.var['gene_symbol'] = a.var_names.astype(str)
        standardize_obs(a, gse, sample_id=gsm, condition=gsm)
        parts.append(a)

    adata = ad.concat(parts, join='outer', merge='first', index_unique=None)
    return finalize(adata, gse)

## 6. Generic 10x MTX-per-sample loader (GSE219201 / GSE242942 / GSE208629)

In [ ]:
def _read_10x_mtx_triplet(mtx, features, barcodes) -> ad.AnnData:
    with gzip.open(mtx, 'rb') as fh:
        X = sio.mmread(fh).tocsr()
    feats = pd.read_csv(features, sep='\t', header=None)
    bars  = pd.read_csv(barcodes, sep='\t', header=None)[0].astype(str).tolist()
    # features.tsv from CellRanger: [ensembl_id, gene_symbol, feature_type]
    var = pd.DataFrame(index=feats[0].astype(str).values)
    var['ensembl_id']  = feats[0].astype(str).values
    var['gene_symbol'] = feats[1].astype(str).values if feats.shape[1] >= 2 else var.index
    if X.shape == (len(var), len(bars)):
        a = ad.AnnData(X=X.T.tocsr(), obs=pd.DataFrame(index=bars), var=var)
    else:
        a = ad.AnnData(X=X.tocsr(), obs=pd.DataFrame(index=bars), var=var)
    return a


def load_10x_mtx_per_sample(gse: str, condition_from_label=None):
    """Generic loader for series with one (matrix.mtx, features.tsv, barcodes.tsv) triplet per GSM."""
    src = prepare_gse(gse)
    mtxs = sorted(src.rglob('*_matrix.mtx.gz'))
    parts = []
    for mtx in tqdm(mtxs, desc=gse):
        prefix = mtx.name[:-len('matrix.mtx.gz')]
        features = mtx.with_name(prefix + 'features.tsv.gz')
        barcodes = mtx.with_name(prefix + 'barcodes.tsv.gz')
        if not features.exists():
            features = mtx.with_name(prefix + 'genes.tsv.gz')
        m = re.match(r'(GSM\d+)_(.+)_$', prefix)
        gsm, label = (m.group(1), m.group(2)) if m else (prefix.split('_')[0], prefix)
        cond = condition_from_label(label) if condition_from_label else label
        a = _read_10x_mtx_triplet(mtx, features, barcodes)
        standardize_obs(a, gse, sample_id=gsm, condition=cond)
        parts.append(a)
    adata = ad.concat(parts, join='outer', merge='first', index_unique=None)
    return finalize(adata, gse)


def load_GSE219201():
    # WT_1 / WT_2 / WT_3 / Phd1-KO_1 / Phd1-KO_2 / Phd1-KO_3
    def cond(label):
        return 'Phd1-KO' if 'KO' in label else 'WT'
    return load_10x_mtx_per_sample('GSE219201', cond)


def load_GSE242942():
    # CON / PF
    def cond(label):
        return 'PF-04457845' if label.startswith('PF') else 'Control'
    return load_10x_mtx_per_sample('GSE242942', cond)


def load_GSE208629():
    # SMA / Con
    def cond(label):
        return 'SMA' if label.upper().startswith('SMA') else 'Control'
    return load_10x_mtx_per_sample('GSE208629', cond)

## 7. Run all & save one `.h5ad` per GSE

In [ ]:
LOADERS = {
    'GSE287569': load_GSE287569,
    'GSE173524': load_GSE173524,
    'GSE167332': load_GSE167332,
    'GSE219201': load_GSE219201,
    'GSE242942': load_GSE242942,
    'GSE208629': load_GSE208629,
}

results = {}
for gse, fn in LOADERS.items():
    print(f'\n=== {gse} ===')
    try:
        adata = fn()
        report(adata, gse)
        out = OUTPUT_DIR / f'{gse}.h5ad'
        adata.write_h5ad(out, compression='gzip')
        print(f'  saved -> {out}  ({out.stat().st_size/1e6:.1f} MB)')
        results[gse] = adata
    except Exception as e:
        print(f'  !! {gse} failed: {e!r}')

## 8. Cross-dataset gene overlap (case-insensitive)
Compares gene symbol sets across all loaded GSEs after upper-casing, so e.g. mouse `Sod1` and human `SOD1` collide intentionally.

In [ ]:
if results:
    sets = {gse: set(a.var['gene_symbol_upper'].astype(str)) for gse, a in results.items()}
    keys = list(sets)
    overlap = pd.DataFrame(index=keys, columns=keys, dtype=int)
    for i in keys:
        for j in keys:
            overlap.loc[i, j] = len(sets[i] & sets[j])
    print('Pairwise gene-symbol overlap (upper-cased):')
    print(overlap.to_string())
    common_all = set.intersection(*sets.values())
    union_all  = set.union(*sets.values())
    print(f'\nCommon across ALL: {len(common_all):,}  |  Union: {len(union_all):,}')

In [ ]:
# Optional: copy outputs into Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_OUT = Path('/content/drive/MyDrive/SMA_h5ad'); DRIVE_OUT.mkdir(parents=True, exist_ok=True)
# for p in OUTPUT_DIR.glob('*.h5ad'):
#     shutil.copy2(p, DRIVE_OUT / p.name)
# print('copied to', DRIVE_OUT)